In [ ]:
import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

In [ ]:
template_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
pdb_ids = template_df.index.to_list()

In [ ]:
ensembles_dir = "../_data/mhcii_tcr_ensembles/"
sampling_methods = ["ensemble/annealing"]
docking_method = "haddock3_rigidbody"

In [ ]:
df = None

for pdb_id in pdb_ids:
    for sampling_method in sampling_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method, docking_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["sampling_method"] = sampling_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

df

In [ ]:
results = dict()

af3_dockqs = pd.read_csv("../data/mhcii_tcr_models/alphafold3/stats.csv", index_col="pdb_id")
tf_dockqs = pd.read_csv("../data/mhcii_tcr_models/tfold-tcr/stats.csv", index_col="pdb_id")

for pdb_id in tqdm(pdb_ids):
    results[pdb_id] = dict()
    sub_df = df[df.pdb_id == pdb_id]
    for sampling_method in df.sampling_method.unique():
        sub_sub_df = sub_df[sub_df.sampling_method == sampling_method]
        results[pdb_id][sampling_method] = sub_sub_df.dockq.values.max()
    results[pdb_id]["alphafold3"] = af3_dockqs.loc[pdb_id].DockQ
    try:
        results[pdb_id]["tfold_tcr"] = tf_dockqs.loc[pdb_id].DockQ
    except:
        results[pdb_id]["tfold_tcr"] = 0

results_df = pd.DataFrame.from_dict(results, orient="index")
results_df

In [ ]:
matrices = list()
traces = dict()
for z in sampling_methods + ["tfold_tcr", "alphafold3"]:
    traces[z] = list()

for pdb_id in pdb_ids:

    for z in ["tfold_tcr", "alphafold3"]:
        traces[z].append(results[pdb_id][z])

    matrix = np.zeros((len(sampling_methods), len(results_df.columns)))

    for i, x in enumerate(sampling_methods):
        score = results_df.loc[pdb_id][x]
        traces[x].append(score)
        for j, y in enumerate(sampling_methods + ["tfold_tcr", "alphafold3"]):
            matrix[i, j] = int(score > results_df.loc[pdb_id][y])

    matrices.append(matrix)

fig = go.Figure(data=go.Heatmap(z=np.array(matrices).mean(axis=0) * 100, x=sampling_methods + ["tfold_tcr", "alphafold3"], y=sampling_methods, texttemplate="%{z:.1f}%"))
fig.update_yaxes(autorange="reversed")
fig.show()

fig = go.Figure()
for t, y in traces.items():
    fig.add_trace(go.Scatter(x=pdb_ids, y=y, mode="markers", name=t))
fig.update_layout(plot_bgcolor="white")
fig.show()

In [ ]:
rank_df = results_df.copy()

for sampling_method in ["ensemble/annealing"]:

    selections = pd.DataFrame(columns=df.columns)

    max_dockqs = list()
    highest_ranked_dockqs = list()
    spearmans = list()
    pdb_ids = list()

    sub_df = df[df.sampling_method == sampling_method]

    for pdb_id in tqdm(sub_df.pdb_id.unique().tolist()):

        pdb_ids.append(pdb_id)

        train_df = sub_df[sub_df.pdb_id != pdb_id]
        test_df = sub_df[sub_df.pdb_id == pdb_id].copy()

        X_train = train_df[["crossing_angle", "incident_angle", "binding_core_sasa"]].values
        y_train = train_df["dockq"].values

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        X_test = test_df[["crossing_angle", "incident_angle", "binding_core_sasa"]].values
        y_test = test_df["dockq"].values
        max_dockqs.append(y_test.max())
        y_hat = model.predict(X_test)
        test_df["y_hat"] = y_hat
        these_selections = test_df.sort_values(by="y_hat", ascending=False)
        selections = pd.concat([selections, these_selections])
        highest_ranked_dockqs.append(these_selections.iloc[0].dockq)

        spearmans.append(spearmanr(y_test, y_hat)[0])

    rank_df[sampling_method] = highest_ranked_dockqs

    fig = px.box(x=spearmans, title=sampling_method, hover_name=pdb_ids)
    fig.update_xaxes(range=(0, 1))
    fig.update_yaxes(range=(-1, 1))
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub_df.pdb_id.unique(), y=max_dockqs, mode="markers"))
    fig.add_trace(go.Scatter(x=sub_df.pdb_id.unique(), y=highest_ranked_dockqs, mode="markers"))
    fig.show()

rank_df

In [ ]:
import numpy as np

matrices = list()
traces = dict()
for z in sampling_methods + ["tfold_tcr", "alphafold3"]:
    traces[z] = list()

for pdb_id in pdb_ids:

    for z in ["tfold_tcr", "alphafold3"]:
        traces[z].append(results[pdb_id][z])

    matrix = np.zeros((len(sampling_methods), len(rank_df.columns)))

    for i, x in enumerate(sampling_methods):
        score = rank_df.loc[pdb_id][x]
        traces[x].append(score)
        for j, y in enumerate(sampling_methods + ["tfold_tcr", "alphafold3"]):
            matrix[i, j] = int(score > rank_df.loc[pdb_id][y])

    matrices.append(matrix)

fig = go.Figure(data=go.Heatmap(z=np.array(matrices).mean(axis=0) * 100, x=sampling_methods + ["tfold_tcr", "alphafold3"], y=sampling_methods, texttemplate="%{z:.1f}%"))
fig.update_yaxes(autorange="reversed")
fig.show()

fig = go.Figure()
for t, y in traces.items():
    fig.add_trace(go.Scatter(x=pdb_ids, y=y, mode="markers", name=t))
fig.update_layout(plot_bgcolor="white")
fig.show()

In [ ]:
selections

In [ ]:
bins = [0, 0.24, 0.51, 0.81, 1.0]

def foo(x):

    for i in range(5):
        if bins[i] > x:
            return i - 1

matrix_1 = np.zeros((6, len(pdb_ids)))
matrix_2 = np.zeros((6, len(pdb_ids)))

for i, pdb_id in enumerate(pdb_ids):
    sub_df = selections[selections.pdb_id == pdb_id].sort_values(by="y_hat", ascending=False)
    ranks = sub_df.dockq.rank(ascending=False).astype(int)
    for j, k in enumerate([1, 5, 10, 20, 50, 100]):
        matrix_1[j, i] = foo(sub_df.head(k).dockq.max())
        matrix_2[j, i] = ranks.head(k).min()

algae = px.colors.sequential.algae

eps = 1e-6
custom_scale = [[0.0, "white"]]

n = len(algae)
for i, color in enumerate(algae):
    t = eps + (1 - eps) * (i / (n - 1))
    custom_scale.append([t, color])

fig = go.Figure(data=go.Heatmap(z=matrix_1, x=pdb_ids, y=[f"top {i}" for i in [1, 5, 10, 20, 50, 100]], colorscale=custom_scale, showscale=False, xgap=1, ygap=1))
fig.update_layout(yaxis_scaleanchor="x")
fig.update_yaxes(domain=[0, 1])
fig.show()

fig = go.Figure(data=go.Heatmap(z=matrix_2, x=pdb_ids, y=[f"top {i}" for i in [1, 5, 10, 20, 50, 100]], colorscale="purp", reversescale=True, xgap=1, ygap=1))
fig.update_layout(yaxis_scaleanchor="x")
fig.update_yaxes(domain=[0, 1])
fig.show()